# 12 — Full-cycle Gemma/Tunix actor + separate critic → hybrid PPO evaluation

Эта тетрадка показывает полный контур: CrafText runtime, MegaPrompts, настоящий Gemma/Tunix actor, separate critic scoring, `HybridPpoStep`, masked PPO boundary, replay evidence и phase profiling. Notebook не скачивает веса: нужен явный local snapshot.


In [ ]:
from pathlib import Path

import jax
import jax.numpy as jnp

from tunix.models.automodel import call_model_config

from tunix_craftext.rollouts.batched import collect_batched_text_rollout, replays_from_batched_rollout
from tunix_craftext.env.config import load_mvp_config
from tunix_craftext.rollouts.hybrid import (
    compute_masked_step_token_ppo_loss,
    hybrid_step_from_text_trajectory,
    last_valid_token_values,
)
from tunix_craftext.artifacts.profiling import PhaseProfiler, save_profile
from tunix_craftext.env.prompts import MegaPromptRenderer
from tunix_craftext.artifacts.replay import save_replay
from tunix_craftext.research.llm_ppo import evaluate_separate_llm_actor_critic_ppo
from tunix_craftext.env.runtime import build_craftext_runtime
from tunix_craftext.env.tasks import CrafTextTaskSampler
from tunix_craftext.artifacts.text_trajectory import text_trajectory_from_replay
from tunix_craftext.models.tunix_actor import build_gemma_tunix_actor, init_linear_value_head


In [ ]:
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / 'pyproject.toml').is_file()), None)
if ROOT is None:
    raise RuntimeError('Run this notebook from inside the tunix-craftext repository')

CONFIG_PATH = ROOT / 'configs/env/smoke/tiny_craftext.yaml'
MODEL_PROFILE = ROOT / 'configs/models/gemma3_270m_instruction.yaml'
SNAPSHOT = ROOT / 'artifacts/models/gemma3-270m-it'
TRAJECTORY_DIR = ROOT / 'artifacts/trajectories/gemma-full-cycle-notebook'
PROFILE_PATH = ROOT / 'artifacts/profiles/gemma-full-cycle-notebook.json'

if not SNAPSHOT.is_dir():
    raise FileNotFoundError(f'No explicit local Gemma snapshot: {SNAPSHOT}')

SEED = 0
BATCH_SIZE = 2
HORIZON = 2
MAX_NEW_TOKENS = 8
CACHE_SIZE = 1024


## 1. Build CrafText runtime, MegaPrompts, Gemma actor and separate critic

`GemmaTunixActor` owns generation and actor token scoring. `actor.critic()` returns a separate critic role over the same Tunix backbone plus value head; the roles merge only at the PPO objective boundary.


In [ ]:
config = load_mvp_config(CONFIG_PATH)
runtime = build_craftext_runtime(config)
task_sampler = CrafTextTaskSampler.from_runtime(
    runtime,
    horizon=config.environment.horizon,
    mode='fixed',
    fixed_instruction_index=config.environment.instruction_index,
)
prepared_goal, prepared_instruction_index = task_sampler.task_at(config.environment.instruction_index)
renderer = MegaPromptRenderer(config.prompt.template)

gemma_config = call_model_config('gemma3_270m_it')
value_head = init_linear_value_head(jax.random.PRNGKey(SEED), hidden_dim=gemma_config.embed_dim)
actor = build_gemma_tunix_actor(
    profile_path=MODEL_PROFILE,
    snapshot=SNAPSHOT,
    cache_size=CACHE_SIZE,
    value_head=value_head,
    seed=SEED,
)
critic = actor.critic()

print('runtime actions:', runtime.actions.labels)
print('actor profile:', actor.profile.model_id)
print('prepared CrafText task:', prepared_instruction_index, prepared_goal)


## 2. Collect batched rollout and export replay evidence

The model sees MegaPrompts output, Tunix generates text/tokens/logprobs, strict decode maps the response to a legal action or audited fallback, and CrafText advances through the adapter.


In [ ]:
profiler = PhaseProfiler()

with profiler.section('rollout'):
    rollout = collect_batched_text_rollout(
        runtime.adapter,
        renderer,
        actor.backend,
        actions=runtime.actions,
        batch_size=BATCH_SIZE,
        horizon=HORIZON,
        seed=SEED,
        goal=prepared_goal,
        max_new_tokens=MAX_NEW_TOKENS,
        invalid_action='fallback',
        fallback_action_id=runtime.actions.index_of('NOOP'),
    )

replays = replays_from_batched_rollout(
    rollout,
    config_path=str(CONFIG_PATH.relative_to(ROOT)),
    commit='notebook',
    backend=f'tunix-gemma:{actor.profile.model_id}',
)
for index, replay in enumerate(replays):
    path = TRAJECTORY_DIR / f'env-{index}.json'
    save_replay(path, replay)
    print('saved replay:', path, 'steps=', len(replay.steps))


## 3. Replay → TextTrajectoryBatch → HybridPpoStep

`TextTrajectoryBatch` is staging evidence. `HybridPpoStep` is the PPO-ready boundary with prompt tokens, generation tokens, rollout logprobs, critic values, actor-learning mask and alive step mask.


In [ ]:
batch = text_trajectory_from_replay(replays[0])
batch.validate()

with profiler.section('actor_scoring'):
    actor_scores = actor.score_actor_tokens(
        prompt_token_ids=batch.prompt_token_ids,
        prompt_token_mask=batch.prompt_token_mask,
        token_ids=batch.token_ids,
        token_mask=batch.token_mask,
    )

with profiler.section('critic_scoring'):
    critic_values = critic.score_values(
        prompt_token_ids=batch.prompt_token_ids,
        prompt_token_mask=batch.prompt_token_mask,
        token_ids=batch.token_ids,
        token_mask=batch.token_mask,
    )

step_values = last_valid_token_values(critic_values.values, batch.token_mask)
hybrid_step = hybrid_step_from_text_trajectory(
    batch,
    values=step_values,
    actor_log_probs=batch.old_logprobs,
)

print('token_ids:', batch.token_ids.shape)
print('hybrid values:', hybrid_step.values.tolist())
print('actor loss tokens:', int(jnp.sum(hybrid_step.actor_loss_token_mask)))
print('step mask:', hybrid_step.step_mask.tolist())


## 4. PPO evaluation over real actor/critic values

This computes two aligned views: full token PPO metrics from real actor/critic scoring, and the hybrid step-token actor loss used to verify post-terminal and fallback masks.


In [ ]:
with profiler.section('ppo_evaluation'):
    evaluation = evaluate_separate_llm_actor_critic_ppo(
        batch,
        actor_scores,
        critic_values,
        learning_mask=hybrid_step.actor_loss_token_mask,
        gamma=0.99,
        clip_epsilon=0.2,
        value_coefficient=0.5,
        entropy_coefficient=0.01,
    )
    step_advantages = jnp.sum(evaluation.advantages * hybrid_step.actor_loss_token_mask, axis=-1) / jnp.maximum(
        jnp.sum(hybrid_step.actor_loss_token_mask, axis=-1),
        1,
    )
    hybrid_actor_loss = compute_masked_step_token_ppo_loss(
        actor_log_probs=actor_scores.token_logprobs,
        old_log_probs=hybrid_step.actor_log_probs,
        generation_token_mask=hybrid_step.actor_loss_token_mask,
        advantages=step_advantages,
        step_valid_mask=hybrid_step.step_mask,
        clip_epsilon=0.2,
    )

evaluation.validate(batch)
assert bool(jnp.isfinite(evaluation.loss))
assert bool(jnp.isfinite(hybrid_actor_loss))
assert tuple(actor_scores.token_logprobs.shape) == tuple(batch.token_ids.shape)
assert tuple(critic_values.values.shape) == tuple(batch.token_ids.shape)

print('token PPO loss:', float(evaluation.loss))
print('hybrid actor loss:', float(hybrid_actor_loss))
print({name: float(value) for name, value in evaluation.metrics.items()})


## 5. Save phase profile

The saved JSON is compatible with the repository profiling/dashboard lane. Accelerator profiling can wrap these same sections with NVIDIA JAX-Toolbox / `nsys-jax`.


In [ ]:
save_profile(
    PROFILE_PATH,
    profiler.events(),
    metadata={
        'notebook': '12_full_cycle_craftext_training.ipynb',
        'model': actor.profile.model_id,
        'batch_size': BATCH_SIZE,
        'horizon': HORIZON,
        'max_new_tokens': MAX_NEW_TOKENS,
    },
)
print('profile saved:', PROFILE_PATH)
for name, event in profiler.summary().items():
    print(name, event)


Итог: notebook строит реальные actor/critic tensors и проверяет их через новый hybrid PPO boundary. Следующий экспериментальный запуск может заменить local single-device actor/critic на Tunix `RLCluster` roles без изменения структуры evidence.
